# DistilBERT implementation

This notebook implements a DistilBERT model for sentiment classification of product reviews. The objective is to fine-tune a pre-trained DistilBERT model and evaluate its performance on the validation set. Different learning rates and numbers of epochs are tested. The best configuration is selected based on validation F1-score. The selected model is then used to generate predictions for the test set.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import torch

from pathlib import Path

from datasets import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

## Load and prepare the data

In [ ]:
DATA_DIR = Path("../data")

train_df = pd.read_json(DATA_DIR / "train.json")
test_df = pd.read_json(DATA_DIR / "test.json")

print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

train_df.head()
train_df.info()
train_df["sentiments"].value_counts()

## Train-validation splits

In [ ]:
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["sentiments"]
)

print("Training set:", train_df.shape)
print("Validation set:", val_df.shape)

print("Training class distribution:")
print(train_df["sentiments"].value_counts(normalize=True))

print("\nValidation class distribution:")
print(val_df["sentiments"].value_counts(normalize=True))

## Model configuration

In [ ]:
MODEL_NAME = "distilbert/distilbert-base-uncased"

bert_configs = [
    {"learning_rate": 2e-5, "epochs": 2},
    {"learning_rate": 1e-5, "epochs": 2},
    {"learning_rate": 2e-5, "epochs": 3},
    {"learning_rate": 1e-5, "epochs": 3}
]

BATCH_SIZE = 16
WEIGHT_DECAY = 0.01
SEED = 42

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

## Tokenization and dataset preparation

Convert pandas DataFrames into Hugging Face Dataset objects.

In [ ]:
train_ds = Dataset.from_pandas(train_df.rename(columns={"sentiments": "label"}), preserve_index=False)
val_ds   = Dataset.from_pandas(val_df.rename(columns={"sentiments": "label"}), preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["reviews"]], preserve_index=False)

print(train_ds)
print(val_ds)
print(test_ds)

def preprocess_function(examples):
    return tokenizer(examples['reviews'], truncation=True)

tokenized_train = train_ds.map(preprocess_function, batched=True)
tokenized_val = val_ds.map(preprocess_function, batched=True)
tokenized_test = test_ds.map(preprocess_function, batched=True)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Evaluation metrics

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    precision, recall, f1, _ = (precision_recall_fscore_support(
          labels,
          predictions,
          average="binary",
          zero_division=0
          ))

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## Model training and hyperparameter selection

In [ ]:
bert_experiment_results = []

best_bert_f1 = -1
best_bert_trainer = None
best_bert_config = None

for i, config in enumerate(bert_configs):

    print(f"\nExperiment {i + 1}/{len(bert_configs)}")
    print("Configuration:", config)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    training_args = TrainingArguments(
        output_dir=f"review_model/experiment_{i + 1}",
        learning_rate=config["learning_rate"],
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=config["epochs"],
        weight_decay=WEIGHT_DECAY,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1",
        greater_is_better=True,
        seed=SEED,
        data_seed=SEED,
        push_to_hub=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    results = trainer.evaluate()

    experiment_result = {
        "experiment": i + 1,
        "learning_rate": config["learning_rate"],
        "epochs": config["epochs"],
        "accuracy": results["eval_accuracy"],
        "precision": results["eval_precision"],
        "recall": results["eval_recall"],
        "f1": results["eval_f1"]
    }

    bert_experiment_results.append(experiment_result)

    print("\nValidation results:")
    print(results)

    if results["eval_f1"] > best_bert_f1:
        best_bert_f1 = results["eval_f1"]
        best_bert_trainer = trainer
        best_bert_config = config

print("\nBest configuration:", best_bert_config)
print("Best validation F1:", best_bert_f1)

# Save the best model and tokenizer for later
best_bert_trainer.save_model("review_model/best_distilbert")
tokenizer.save_pretrained("review_model/best_distilbert")
print("Best DistilBERT model saved.")


## Comparison of DistilBERT configurations

In [ ]:
bert_results_df = pd.DataFrame(bert_experiment_results)

bert_results_df = bert_results_df.sort_values(
    by="f1",
    ascending=False
).reset_index(drop=True)

bert_results_df

print("Best DistilBERT configuration:")
print(best_bert_config)

print("\nBest validation F1-score:")
print(best_bert_f1)

## Evaluation of the best model

In [ ]:
val_output = best_bert_trainer.predict(tokenized_val)

val_predictions = np.argmax(
    val_output.predictions,
    axis=1
)

val_labels = val_output.label_ids

print(classification_report(
        val_labels,
        val_predictions,
        zero_division=0))

In [ ]:
cm = confusion_matrix(
    val_labels,
    val_predictions
)

print("Confusion matrix:")
print(cm)


## Prediction confidence analysis

In [ ]:
## Test set predictions and confidence analysis

test_ds = Dataset.from_pandas(
    test_df[["reviews"]],
    preserve_index=False
)

tokenized_test = test_ds.map(preprocess_function, batched=True)

final_trainer = Trainer(
    model=best_bert_trainer.model,
    processing_class=tokenizer,
    data_collator=data_collator
)

test_logits = final_trainer.predict(tokenized_test).predictions
test_predictions = np.argmax(test_logits, axis=1)

# Generate submission file
submission_df = pd.DataFrame({
    "reviews": test_df["reviews"],
    "sentiments": test_predictions
})

submission_df.to_csv("submission.csv", index=False)

print("\nTEST SET PREDICTIONS")
print(f"Test samples: {len(test_predictions)}")
print(f"Predicted positive (1): {(test_predictions == 1).sum()}")
print(f"Predicted negative (0): {(test_predictions == 0).sum()}")

# Calculate prediction confidence
test_confidence = torch.softmax(
    torch.tensor(test_logits),
    dim=1
).max(dim=1).values.numpy()

analysis_df = pd.DataFrame({
    "reviews": test_df["reviews"],
    "prediction": test_predictions,
    "confidence": test_confidence.round(2)
})

pd.set_option("display.max_colwidth", 400)

print("\nLEAST CONFIDENT")
display(analysis_df.sort_values("confidence").head(10))

print("\nMOST CONFIDENT")
display(analysis_df.sort_values("confidence", ascending=False).head(10))